# Optimization

Use gpu=ptxinfo

## I. Host and device

Minimize this, cause they are slow

### a. Pinned mem

Reduction in using pinned vs pageable

```-query-gpu=pcie.gen.current,pcie.link.gen.max```

- PCIe link is often the bottleneck for data transfers between CPU memory and GPU memory

- pageable <<< pinned -> transfer size vs bandwidth

- One big transfer better than many small -> up to some point

In [ ]:
!nvfortran opt/1_al.cuf
!./a.out

## II. Device mem

- Global -> device atr
- Local, regs -> on-chip regs, per-thread access
- Shared mem -> inside thread block
- Constant -> with constant atr

![Mem](imgs/1.png)

#### a. Global mem

- Assumed sizes: a(nx, ny, *)

- Assumed shapes: a(:, :, :) -> from array passed : use more registers (in every thread)

##### Thread warps

All data transactions are 32, 64 or 128 byte mem -> contiguos

- Offsets dont change that -> still contiguos : offten add 1 (2:1)

- Strides do change that -> not contiguos : offten add 32 (32:1)

- Cache line is 128 bytes

- All threads in warp execute at the same time -> mem trans per warp

- The offset may cross the 128 byte boundary

##### Access flow

Thread -> L1 -> L2 ->  DRAM

If not in L1, warps need to switch from L1 and bring from DRAM

In [ ]:
#offset
!nvfortran opt/2_glob.cuf
!./a.out

In [ ]:
#stride
!nvfortran opt/3_str.cuf
!./a.out

#### b. Local mem

Is expensive as global mem. Private for a specific thread. Large structure or arrays.

- automatic = global

Regs can be decided to be used instead -> small data

Regs -> Local Mem <- Stack Frame

In [ ]:
!nvfortran opt/4_local.cuf
!./a.out

#### c. Constant mem

In device -> but also cache in each sm : read by kernel -> read/write by host

In [ ]:
!nvfortran opt/5_const.cuf
!./a.out

#### d. L1 and L2 caches

Each sm has L1, L2 shared by all sms -> glob mem and local is cached.

- Per sm block
- gpu=loadache:[L1|L2] -> :L1 or :L2

In [ ]:
!nvfortran opt/6_cache.cuf
!./a.out

#### e. Shared mem

Shared in thread block -> per thread block

- Unified: shared mem + L1, many shared mem, only one L1

- Dynamic and static (depends on max allocation)

##### Coalescing glob mem (adj access)

- Make a copy in shared mem, which doesnt need to be coalesced

-> Compare: Copy mat, copy shared mem, transpose (strided), transpose (coalesced)

- Ex: Transpose

- Fort -> Coalesced: row based

In [ ]:
!nvfortran opt/7_coal.cuf
!./a.out

Divided into banks -> accessed sim. In warp, threadID%y is the same.

- 2D(1, 32)

Banks:

- (0,0) (0,1) ... -> diff rows (parallel)
- (1,0)  (1,1)...
- (2,0 ) (2,1)...

Global Mem

- (0,0) (1,0) -> same row (coalesced): 1 access at the time
- (0,1) (1,1)
- (0,2) (1,2)

-> Better access: row access

![bank1](imgs/2.png)
![bank2](imgs/3.png)

#### f. Regs

- Accesible for threads -> 64K per multiprocessor
- ```-gpu=maxregcount:n```

- Max thr x block
- Min blck x sm
- max blck x clus

## III. Exec config

Achieve parallelism -> concurrent threads, ind ops

#### a. Thread level 

-> How many threads launched. Rely on this: small ocuppancy

- ```cudaOccupancyMaxActiveBlocksPerMultiprocessor()```
- ```cudaDeviceProperties()```

+ threads -> more occu : shared mem is per thread block, regs is per thread

-> max threads per block: 1024, max threads per sm: 2024

#### b. Instr level

One thread many instructions

-> local variables to threads -> regs

-> dis: private data in regs

In [ ]:
!nvfortran opt/8_ilp.cuf
!./a.out

##### Glob mem to shared

- ```call pipelineMemcpyAsync(aShared(is), aGlobal(i))```
- ```call pipelineCommit()```
- ```call pipelineWaitPrior(D)```

In [ ]:
!nvfortran opt/9_globToshr.cuf
!./a.out

## IV. Inst opt

#### A. Div warps

- All threads in warp must do op if some in lane does -> but some are masked

- Tip: scatter array, and then re group it
